[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/02.%20The%20Container%20Stacking%20Rules%20Problem/P2-Tier-8_executed.ipynb)

---


# The Container Stacking Rules Problem

## Tier 8 — Value-aligned selection framework

Operational efficiency is important, but terminal decisions can also affect service equity, worker workload, and environmental impact. This notebook demonstrates a selection layer that evaluates feasible stacking plans using explicit proxy metrics and a fairness constraint.

### Purpose and workflow

The input is a small set of containers, each with a customer label and a departure time. The notebook enumerates feasible assignments of arriving containers to stacks, measures each plan, filters plans by a fairness threshold, and selects a plan using a weighted score. It then reports and visualizes the stakeholder-level implications.

The solution method is exhaustive candidate enumeration followed by metric evaluation, constraint-based filtering, and score-based selection. It is practical for this small teaching instance; the number of assignments grows exponentially with the number of containers.

### Learning goals

- Translate stated values into measurable, auditable proxies.
- Distinguish a hard feasibility condition from a score penalty.
- Follow how blocking pairs determine the reshuffle and retrieval-cost metrics.
- Interpret the trade-off between reshuffles, fairness, stack-height balance, and the environmental proxy.

### Outputs

- A table of feasible candidate plans and their scores.
- A comparison between the minimum-reshuffle plan and the selected fairness-feasible plan.
- Per-container and per-customer retrieval-cost reports, score components, and plan visualizations.

### Scope and modeling assumptions

Containers arrive in the listed order and are placed in one of a fixed number of stacks, subject to a maximum stack height. Departure times are ordered numeric values. A blocking pair is evaluated within a stack: a lower container is blocked when a container above it departs later. Each such pair contributes one extra move to the lower container's retrieval-cost proxy.

The customer labels and the welfare and environmental measures are illustrative. Stack-height variance is only a workload-balance proxy, and reshuffle count is only an environmental-impact proxy; neither is a direct measurement of worker safety, fuel use, or emissions. The model does not represent crane scheduling, uncertainty, container dimensions, safety rules, contractual priorities, or the full sequence of real retrieval operations. The fairness threshold is a policy choice, not a universal definition of equitable service.

In [ ]:
# Import the libraries used for enumeration, tabulation, and visualization.
from typing import Dict, List, Optional, Tuple

try:
    import itertools
    import math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib "
        "in your Python environment before running this notebook."
    ) from exc

print("Dependencies imported successfully.")

## Concrete ethical scenario

The example contains three customer groups: Premium, Standard, and Budget. Each container has a departure time, represented by `d`; smaller values indicate earlier departures.

The question is not whether a customer group should receive preferential treatment. Instead, the example asks how a measurable service-level proxy can reveal differences between groups, and how a stated fairness threshold can restrict the plans considered for selection. The resulting metrics support an auditable decision, but they do not by themselves establish that a policy is ethically adequate.

In [ ]:
# Each record gives a container identifier, customer group, and departure time.
containers = [
    {"id": "P1", "customer": "Premium", "d": 1},
    {"id": "P2", "customer": "Premium", "d": 4},
    {"id": "P3", "customer": "Premium", "d": 7},
    {"id": "S1", "customer": "Standard", "d": 2},
    {"id": "S2", "customer": "Standard", "d": 5},
    {"id": "B1", "customer": "Budget", "d": 3},
    {"id": "B2", "customer": "Budget", "d": 6},
]

NUM_STACKS = 3
MAX_TIERS = 3

cid_to_d = {container["id"]: container["d"] for container in containers}
cid_to_customer = {
    container["id"]: container["customer"] for container in containers
}
arrival_order = [container["id"] for container in containers]

arrival_order

In [ ]:
def count_blocking_pairs_in_stack(
    stack_bottom_to_top: List[str],
    departure_times: Dict[str, int],
) -> int:
    """Count lower containers blocked by later-departing containers above them."""
    reshuffles = 0
    for lower_index in range(len(stack_bottom_to_top)):
        for upper_index in range(lower_index + 1, len(stack_bottom_to_top)):
            lower = stack_bottom_to_top[lower_index]
            upper = stack_bottom_to_top[upper_index]
            if departure_times[lower] < departure_times[upper]:
                reshuffles += 1
    return reshuffles


def total_reshuffles(
    stack_config: List[List[str]],
    departure_times: Dict[str, int],
) -> int:
    """Sum blocking-pair counts across all stacks in a plan."""
    return sum(
        count_blocking_pairs_in_stack(stack, departure_times)
        for stack in stack_config
    )


total_reshuffles([["P1", "P2"], [], []], cid_to_d)

## Fairness metric: service-level equity

A fairness measure must be defined before it can be calculated or constrained. Here, each container receives a retrieval-cost proxy equal to one base retrieval move plus the number of blocking pairs assigned to that container. A blocking pair is charged to the lower container because retrieving it requires moving the later-departing container above it.

For each customer group, the proxy is averaged across that group's containers. The relative gap compares the largest and smallest group averages, using the smallest average as the reference:

$$
g = \frac{\max_{c \in C} \bar{r}_c - \min_{c \in C} \bar{r}_c}{\min_{c \in C} \bar{r}_c}.
$$

Here, $C$ is the set of customer groups and $\bar{r}_c$ is the average retrieval-cost proxy for group $c$. The base move makes every container's proxy at least one, so the denominator is positive. The policy threshold requires $g \leq \tau$, where $\tau$ is `FAIRNESS_REL_GAP_MAX`. This relative gap is a chosen proxy for differences in average service cost; it is not a complete measure of fairness.

## Step 1 — Define value metrics

Each candidate plan is evaluated on four quantities:

- **Efficiency:** total blocking pairs, interpreted as reshuffles.
- **Fairness:** the relative gap between customer groups' average retrieval-cost proxies.
- **Worker-welfare proxy:** the variance of stack heights. Lower variance means stack heights are more alike, but this does not directly measure workload or safety.
- **Environmental proxy:** the number of extra moves implied by blocking pairs. This is deliberately the same count as the reshuffle metric, so it represents an additional policy weight on extra moves, not an independent environmental measurement.

The hard fairness condition determines which plans are eligible for the value-based selection. A weighted score ranks eligible plans and also assigns a penalty to fairness violations when scores are inspected across all candidates.

In [ ]:
FAIRNESS_REL_GAP_MAX = 0.20
FAIRNESS_NUMERICAL_TOLERANCE = 1e-12


def blocking_pairs(
    stack_config: List[List[str]],
    departure_times: Dict[str, int],
) -> pd.DataFrame:
    """Return one record for each blocking pair in a stacking plan."""
    pairs = []
    for stack_index, stack in enumerate(stack_config, start=1):
        for lower_index in range(len(stack)):
            for upper_index in range(lower_index + 1, len(stack)):
                lower = stack[lower_index]
                upper = stack[upper_index]
                if departure_times[lower] < departure_times[upper]:
                    pairs.append(
                        {
                            "stack": stack_index,
                            "lower": lower,
                            "upper": upper,
                            "lower_departure": departure_times[lower],
                            "upper_departure": departure_times[upper],
                        }
                    )
    return pd.DataFrame(
        pairs,
        columns=[
            "stack",
            "lower",
            "upper",
            "lower_departure",
            "upper_departure",
        ],
    )


def reshuffles_needed_per_container(
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
) -> Dict[str, int]:
    """Assign each blocking-pair move to the lower container it blocks."""
    counts = {container["id"]: 0 for container in containers_data}
    pairs = blocking_pairs(stack_config, departure_times)
    for lower in pairs["lower"]:
        counts[str(lower)] += 1
    return counts


def retrieval_costs_by_customer(
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
) -> pd.DataFrame:
    """Build per-container retrieval-cost proxies for a stacking plan."""
    reshuffles = reshuffles_needed_per_container(
        stack_config, containers_data, departure_times
    )
    rows = []
    for container in containers_data:
        container_id = container["id"]
        rows.append(
            {
                "container": container_id,
                "customer": container["customer"],
                "departure_time": container["d"],
                "reshuffles_needed": reshuffles[container_id],
                "retrieval_cost": 1 + reshuffles[container_id],
            }
        )
    return pd.DataFrame(rows)


def fairness_metrics(
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
    relative_gap_max: float,
) -> Dict:
    """Calculate group averages, relative gap, and threshold status."""
    costs = retrieval_costs_by_customer(
        stack_config, containers_data, departure_times
    )
    averages = costs.groupby("customer")["retrieval_cost"].mean().sort_values()
    min_average = float(averages.min())
    max_average = float(averages.max())
    absolute_gap = max_average - min_average
    relative_gap = absolute_gap / min_average
    fairness_ok = relative_gap <= relative_gap_max + FAIRNESS_NUMERICAL_TOLERANCE

    return {
        "avg_costs": averages,
        "min_avg": min_average,
        "max_avg": max_average,
        "abs_gap": float(absolute_gap),
        "rel_gap": float(relative_gap),
        "fairness_ok": bool(fairness_ok),
        "fairness_violation": float(max(0.0, relative_gap - relative_gap_max)),
    }


def worker_welfare_proxy(stack_config: List[List[str]]) -> float:
    """Measure variation in stack heights as a workload-balance proxy."""
    heights = np.array([len(stack) for stack in stack_config], dtype=float)
    return float(np.var(heights))


def environment_proxy_extra_moves(
    stack_config: List[List[str]],
    departure_times: Dict[str, int],
) -> int:
    """Use the blocking-pair count as a proxy for extra moves."""
    return total_reshuffles(stack_config, departure_times)


print("Metric helpers are ready.")

## Step 2 — Enumerate and score candidate plans

A choice sequence assigns each arriving container to one stack. With $n$ containers and $m$ stacks, there are $m^n$ possible sequences before capacity checks. Each sequence is converted into a bottom-to-top stack configuration; a sequence is infeasible if it places more than `MAX_TIERS` containers in any stack.

For a feasible plan $x$, define the score

$$
J(x) = w_{\mathrm{eff}} R(x)
      + w_{\mathrm{worker}} W(x)
      + w_{\mathrm{env}} E(x)
      + w_{\mathrm{fair}} V(x),
$$

where $R(x)$ is the reshuffle count, $W(x)$ is stack-height variance, $E(x)$ is the extra-move proxy, and $V(x)=\max(0,g(x)-\tau)$ is the fairness violation. The weights are policy parameters. In this example, $E(x)=R(x)$, so the efficiency and environmental terms intentionally place two weights on the same extra-move count; they should not be interpreted as independent evidence.

Exhaustive enumeration makes every candidate in this small instance inspectable. Its exponential growth limits its use to small instances, which is why the notebook keeps the example deliberately small.

In [ ]:
def apply_choice_sequence(
    choice_seq: Tuple[int, ...],
    arrival_order_data: List[str],
    num_stacks: int,
    max_tiers: int,
) -> Optional[List[List[str]]]:
    """Convert stack choices into a feasible bottom-to-top configuration."""
    if len(choice_seq) != len(arrival_order_data):
        raise ValueError("Each arriving container must have exactly one stack choice.")

    stacks: List[List[str]] = [[] for _ in range(num_stacks)]
    for container_id, stack_index in zip(arrival_order_data, choice_seq):
        if len(stacks[stack_index]) >= max_tiers:
            return None
        stacks[stack_index].append(container_id)
    return stacks


W_EFF = 1.0
W_WORKER = 0.5
W_ENV = 0.75
W_FAIR_VIOL = 50.0


def candidate_scores(
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
    relative_gap_max: float,
) -> Dict:
    """Evaluate the objective components and fairness status of one plan."""
    efficiency = total_reshuffles(stack_config, departure_times)
    fairness = fairness_metrics(
        stack_config, containers_data, departure_times, relative_gap_max
    )
    worker = worker_welfare_proxy(stack_config)
    environment = environment_proxy_extra_moves(stack_config, departure_times)

    total_score = (
        W_EFF * float(efficiency)
        + W_WORKER * worker
        + W_ENV * float(environment)
        + W_FAIR_VIOL * fairness["fairness_violation"]
    )

    return {
        "eff_reshuffles": int(efficiency),
        "fair_rel_gap": fairness["rel_gap"],
        "fair_ok": fairness["fairness_ok"],
        "worker_height_var": worker,
        "env_extra_moves": int(environment),
        "score": float(total_score),
    }


all_sequences = list(
    itertools.product(range(NUM_STACKS), repeat=len(arrival_order))
)
rows = []
feasible_count = 0

for sequence in all_sequences:
    configuration = apply_choice_sequence(
        sequence, arrival_order, NUM_STACKS, MAX_TIERS
    )
    if configuration is None:
        continue

    feasible_count += 1
    scores = candidate_scores(
        configuration,
        containers,
        cid_to_d,
        FAIRNESS_REL_GAP_MAX,
    )
    rows.append(
        {"seq": sequence, "stack_config": configuration, **scores}
    )

candidates_df = pd.DataFrame(rows).sort_values(
    ["eff_reshuffles", "fair_rel_gap", "worker_height_var", "score"],
    ascending=True,
).reset_index(drop=True)

assert len(candidates_df) == feasible_count
assert all(
    sorted(container for stack in config for container in config)
    == sorted(arrival_order)
    for config in candidates_df["stack_config"]
)

print("Total choice sequences:", len(all_sequences))
print("Feasible candidates:", feasible_count)

candidates_df.head(10)

## Step 3 — Select an efficiency-driven plan and a fairness-feasible plan

The efficiency-driven plan minimizes the reshuffle count; ties are ordered by score, fairness gap, and stack-height variance. The value-aligned plan is selected only from candidates satisfying the fairness threshold, and it minimizes the weighted score $J(x)$; ties are resolved by reshuffle count.

This is a constrained selection rule: fairness determines eligibility, while the score ranks eligible candidates. The large fairness-violation weight remains useful in the all-candidate score breakdown, but it does not replace the explicit threshold filter. If no candidate satisfies the threshold, the notebook reports the least-unfair candidate as a fallback rather than describing it as fairness-feasible.

In [ ]:
minimum_reshuffles = int(candidates_df["eff_reshuffles"].min())

best_eff_row = (
    candidates_df[candidates_df["eff_reshuffles"] == minimum_reshuffles]
    .sort_values(
        ["score", "fair_rel_gap", "worker_height_var"],
        ascending=True,
    )
    .iloc[0]
)
best_eff_cfg = best_eff_row["stack_config"]

fair_df = candidates_df[candidates_df["fair_ok"]].copy()

if fair_df.empty:
    best_eth_row = candidates_df.sort_values(
        ["fair_rel_gap", "eff_reshuffles", "score"],
        ascending=True,
    ).iloc[0]
    fairness_note = (
        "No candidate satisfies the fairness threshold; "
        "the least-unfair candidate is shown as a fallback."
    )
else:
    best_eth_row = fair_df.sort_values(
        ["score", "eff_reshuffles"], ascending=True
    ).iloc[0]
    fairness_note = "Selected from candidates satisfying the fairness threshold."

best_eth_cfg = best_eth_row["stack_config"]

print("Efficiency-driven plan:")
print("- Reshuffles:", int(best_eff_row["eff_reshuffles"]))
print("- Fairness relative gap:", float(best_eff_row["fair_rel_gap"]))
print("- Satisfies fairness threshold:", bool(best_eff_row["fair_ok"]))
print("- Score:", float(best_eff_row["score"]))

print("\nValue-aligned plan:")
print("- Selection status:", fairness_note)
print("- Reshuffles:", int(best_eth_row["eff_reshuffles"]))
print("- Fairness relative gap:", float(best_eth_row["fair_rel_gap"]))
print("- Satisfies fairness threshold:", bool(best_eth_row["fair_ok"]))
print("- Score:", float(best_eth_row["score"]))

if not fair_df.empty:
    top_snapshot = fair_df.sort_values(
        ["score", "eff_reshuffles"], ascending=True
    ).head(10)
else:
    top_snapshot = candidates_df.sort_values(
        ["fair_rel_gap", "eff_reshuffles", "score"], ascending=True
    ).head(10)

top_snapshot[
    [
        "eff_reshuffles",
        "fair_rel_gap",
        "fair_ok",
        "worker_height_var",
        "env_extra_moves",
        "score",
        "seq",
    ]
]

## Step 4 — Explainability: report stakeholder impacts

For each selected plan, the report includes its stack configuration, reshuffle count, fairness gap, and threshold status. It also returns a per-customer table of average retrieval-cost proxies and a per-container table showing how blocking pairs contribute to those costs.

These quantities make the modeled decision auditable: a reader can trace the group averages back to container-level values and blocking pairs. They should be interpreted as outputs of the stated proxies, not as direct measurements of actual service quality.

In [ ]:
def report_plan(
    name: str,
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
    relative_gap_max: float,
}:
    """Print plan-level measures and return group and container reports."""
    fairness = fairness_metrics(
        stack_config, containers_data, departure_times, relative_gap_max
    )

    print("=" * 80)
    print(name)
    print("Stack configuration:", stack_config)
    print("Reshuffles:", total_reshuffles(stack_config, departure_times))
    print("Fairness relative gap:", fairness["rel_gap"])
    print("Satisfies fairness threshold:", fairness["fairness_ok"])

    average_df = fairness["avg_costs"].to_frame("avg_retrieval_cost")
    per_container_df = retrieval_costs_by_customer(
        stack_config, containers_data, departure_times
    ).sort_values(["customer", "departure_time"])

    return average_df, per_container_df


avg_eff, per_eff = report_plan(
    "Efficiency-driven plan",
    best_eff_cfg,
    containers,
    cid_to_d,
    FAIRNESS_REL_GAP_MAX,
)
avg_eth, per_eth = report_plan(
    "Value-aligned plan",
    best_eth_cfg,
    containers,
    cid_to_d,
    FAIRNESS_REL_GAP_MAX,
)

avg_eff, avg_eth

## Step 5 — Visual comparison

The stack diagrams show containers from bottom to top and label each container with its customer group and departure time. An arrow points from an upper container toward the lower container it blocks. A second plot compares average retrieval-cost proxies by customer for the two selected plans.

The diagrams show the modeled blocking relationships, while the bar chart shows how those relationships affect the group averages. Neither visualization represents a full operational timeline.

In [ ]:
def plot_stack_with_blocking(
    stack_config: List[List[str]],
    title: str,
    customer_by_id: Dict[str, str],
    departure_times: Dict[str, int],
    num_stacks: int,
    max_tiers: int,
) -> None:
    """Draw stack positions and arrows for all modeled blocking pairs."""
    pairs = blocking_pairs(stack_config, departure_times)
    figure, axis = plt.subplots(figsize=(9, 4))

    stack_width = 1.0
    cell_height = 1.0
    x_gap = 0.6
    stack_x = {
        stack_index: (stack_index - 1) * (stack_width + x_gap)
        for stack_index in range(1, num_stacks + 1)
    }

    for stack_index in range(1, num_stacks + 1):
        x_position = stack_x[stack_index]
        for tier in range(max_tiers):
            rectangle = plt.Rectangle(
                (x_position, tier * cell_height),
                stack_width,
                cell_height,
                facecolor="#F2F4F7",
                edgecolor="#344054",
            )
            axis.add_patch(rectangle)

    for stack_index, stack in enumerate(stack_config, start=1):
        x_position = stack_x[stack_index]
        for tier, container_id in enumerate(stack):
            customer = customer_by_id[container_id]
            departure = departure_times[container_id]
            axis.text(
                x_position + stack_width / 2,
                tier * cell_height + cell_height / 2,
                f"{container_id}\n({customer})\nd={departure}",
                ha="center",
                va="center",
                fontsize=9,
                color="#101828",
            )

    for _, pair in pairs.iterrows():
        stack_index = int(pair["stack"])
        stack = stack_config[stack_index - 1]
        lower_tier = stack.index(str(pair["lower"]))
        upper_tier = stack.index(str(pair["upper"]))
        x_position = stack_x[stack_index] + stack_width / 2
        lower_y = lower_tier * cell_height + cell_height / 2
        upper_y = upper_tier * cell_height + cell_height / 2

        axis.annotate(
            "",
            xy=(x_position, lower_y),
            xytext=(x_position, upper_y),
            arrowprops={"arrowstyle": "->", "lw": 2, "color": "#F97066"},
        )

    axis.set_title(title)
    axis.set_xlabel("Stack")
    axis.set_ylabel("Tier (bottom to top)")
    axis.set_xticks(
        [stack_x[index] + stack_width / 2 for index in range(1, num_stacks + 1)]
    )
    axis.set_xticklabels([f"Stack {index}" for index in range(1, num_stacks + 1)])
    axis.set_yticks(range(max_tiers))
    axis.set_yticklabels([str(tier + 1) for tier in range(max_tiers)])
    axis.set_xlim(-0.5, stack_x[num_stacks] + stack_width + 0.5)
    axis.set_ylim(-0.2, max_tiers * cell_height + 0.2)
    plt.tight_layout()
    plt.show()


plot_stack_with_blocking(
    best_eff_cfg,
    "Efficiency-driven plan",
    cid_to_customer,
    cid_to_d,
    NUM_STACKS,
    MAX_TIERS,
)
plot_stack_with_blocking(
    best_eth_cfg,
    "Value-aligned plan",
    cid_to_customer,
    cid_to_d,
    NUM_STACKS,
    MAX_TIERS,
)

average_comparison = pd.concat(
    [
        avg_eff.rename(columns={"avg_retrieval_cost": "efficiency_driven"}),
        avg_eth.rename(columns={"avg_retrieval_cost": "value_aligned"}),
    ],
    axis=1,
)

axis = average_comparison.plot(
    kind="bar",
    figsize=(8, 3.2),
    rot=0,
    edgecolor="#101828",
)
axis.set_title("Average retrieval-cost proxy by customer (lower is better)")
axis.set_xlabel("Customer")
axis.set_ylabel("Average retrieval-cost proxy (moves)")
axis.legend(title="Plan")
axis.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

average_comparison

## Additional analysis — Make score components explicit

A total score is easier to audit when each contribution is visible. The table below separates the efficiency, worker-proxy, environmental-proxy, and fairness-violation terms for the selected plans. Because the environmental proxy is defined as the reshuffle count, its score contribution is intentionally correlated with the efficiency contribution.

In [ ]:
def score_breakdown(
    stack_config: List[List[str]],
    containers_data: List[Dict],
    departure_times: Dict[str, int],
    relative_gap_max: float,
) -> Dict[str, float]:
    """Return the objective components used to evaluate a stacking plan."""
    efficiency = float(total_reshuffles(stack_config, departure_times))
    fairness = fairness_metrics(
        stack_config, containers_data, departure_times, relative_gap_max
    )
    worker = worker_welfare_proxy(stack_config)
    environment = float(
        environment_proxy_extra_moves(stack_config, departure_times)
    )
    fairness_violation = fairness["fairness_violation"]

    efficiency_term = W_EFF * efficiency
    worker_term = W_WORKER * worker
    environment_term = W_ENV * environment
    fairness_term = W_FAIR_VIOL * fairness_violation

    return {
        "eff_reshuffles": efficiency,
        "fair_rel_gap": fairness["rel_gap"],
        "fairness_violation": fairness_violation,
        "worker_height_var": worker,
        "env_extra_moves": environment,
        "term_eff": efficiency_term,
        "term_worker": worker_term,
        "term_env": environment_term,
        "term_fair_violation": fairness_term,
        "score": efficiency_term + worker_term + environment_term + fairness_term,
    }


breakdown_efficiency = score_breakdown(
    best_eff_cfg, containers, cid_to_d, FAIRNESS_REL_GAP_MAX
)
breakdown_value_aligned = score_breakdown(
    best_eth_cfg, containers, cid_to_d, FAIRNESS_REL_GAP_MAX
)

breakdown_df = pd.DataFrame(
    [breakdown_efficiency, breakdown_value_aligned],
    index=["efficiency_driven", "value_aligned"],
).round(4)

assert math.isclose(
    breakdown_efficiency["score"],
    float(best_eff_row["score"]),
    rel_tol=1e-12,
    abs_tol=1e-12,
)
assert math.isclose(
    breakdown_value_aligned["score"],
    float(best_eth_row["score"]),
    rel_tol=1e-12,
    abs_tol=1e-12,
)

breakdown_df

## Interpretation, strengths, and limitations

This example separates two decisions: which plans meet the fairness threshold, and which eligible plan has the lowest weighted score. The minimum-reshuffle plan is reported separately so that operational efficiency can be compared with the fairness-constrained selection. The outputs should be read through the model's definitions: a fairness gap is a difference in group-average retrieval-cost proxies, stack-height variance is a workload-balance proxy, and the environmental proxy counts the same extra moves as the reshuffle metric.

### Strengths

- The metrics, threshold, weights, and selected plan are visible and auditable.
- The fairness condition is an explicit eligibility rule rather than an interpretation of a weighted score alone.
- Candidate enumeration makes the decision process inspectable for this small instance.

### Limitations

- Metrics and thresholds require stakeholder agreement and validation against real outcomes.
- A fairness constraint can exclude the plan with the fewest reshuffles.
- Proxy measures can omit important impacts, and multiple weights on correlated measures can give those impacts additional influence.
- Exhaustive enumeration grows exponentially and is suitable here because the example is small.

In a real terminal, the same modeling structure would require carefully validated data, operational constraints, governance, and a reasoned choice of fairness definitions. Those real-world extensions are outside this runnable example.